#### Install websockets library

In [ ]:
! pip install websockets

#### Put your API key and run

In [3]:
import asyncio
import json
import websockets

API_KEY = "YOUR_API_KEY"  # Replace with your actual key
ENDPOINT = "wss://stream.aisstream.io/v0/stream"

# High-density bounding box: Strait of Gibraltar
# BOUNDING_BOXES = [[[35.8, -6.0], [36.3, -5.0]]]
# The Strait of Malacca & Singapore Strait
BOUNDING_BOXES = [[[1.15, 103.50], [1.45, 104.30]]]


def extract_comprehensive_ais_data(raw_json: dict) -> dict:
    """Extracts and flattens all analytically valuable fields from an AIS message."""
    msg_type = raw_json.get("MessageType", "Unknown")
    meta = raw_json.get("MetaData", {})
    message_content = raw_json.get("Message", {}).get(msg_type, {})

    # 1. Core Header & Spatial Metadata (Always present across all message types)
    extracted = {
        "message_type": msg_type,
        "mmsi": meta.get("MMSI"),
        "ship_name_meta": meta.get("ShipName", "").strip(),
        "latitude": meta.get("latitude"),
        "longitude": meta.get("longitude"),
        "timestamp_utc": meta.get("time_utc"),
    }

    # 2. Extract Dynamic Telemetry (Position Reports: Message Types 1, 2, 3, 18)
    if msg_type in [
        "PositionReport",
        "StandardClassBPositionReport",
        "ExtendedClassBPositionReport",
    ]:
        extracted.update(
            {
                "speed_over_ground_knots": message_content.get("Sog"),
                "course_over_ground_degrees": message_content.get("Cog"),
                "true_heading_degrees": message_content.get("TrueHeading"),
                "navigational_status_code": message_content.get(
                    "NavigationalStatus"
                ),
                "rate_of_turn": message_content.get("RateOfTurn"),
                "position_accuracy": message_content.get("PositionAccuracy"),
                "raim_flag": message_content.get("Raim"),
                "valid_position": message_content.get("Valid"),
            }
        )

    # 3. Extract Static Vessel & Voyage Metadata (Ship Static Data: Message Type 5)
    elif msg_type == "ShipStaticData":
        dimension = message_content.get("Dimension", {})
        eta = message_content.get("Eta", {})

        # Calculate total dimensions in meters if available
        length_meters = None
        width_meters = None
        if dimension:
            length_meters = dimension.get("A", 0) + dimension.get("B", 0)
            width_meters = dimension.get("C", 0) + dimension.get("D", 0)

        extracted.update(
            {
                "imo_number": message_content.get("ImoNumber"),
                "call_sign": message_content.get("CallSign", "").strip(),
                "vessel_name_static": message_content.get("Name", "").strip(),
                "vessel_type_code": message_content.get("Type"),
                "destination": message_content.get("Destination", "").strip(),
                "maximum_present_static_draft": message_content.get(
                    "MaximumStaticDraught"
                ),
                "length_meters": length_meters,
                "width_meters": width_meters,
                "dimension_offset": dimension,
                "eta_details": eta,
                "eta_parsed": (
                    f"Month: {eta.get('Month')}, Day: {eta.get('Day')}, "
                    f"Hour: {eta.get('Hour')}:{eta.get('Minute')}"
                    if eta
                    else None
                ),
                "position_fix_type": message_content.get("FixType"),
            }
        )

    # 4. Fallback for any other AIS message types (Type 4, 21, 24, etc.)
    else:
        extracted["raw_payload"] = message_content

    return extracted


async def run_full_extraction():
    print(f"Connecting to {ENDPOINT}...")

    async with websockets.connect(
        ENDPOINT, ping_interval=20, ping_timeout=20, close_timeout=10
    ) as websocket:

        # Receive ALL AIS message types by excluding "FilterMessageTypes"
        subscribe_payload = {
            "APIKey": API_KEY,
            "BoundingBoxes": BOUNDING_BOXES,
        }

        await websocket.send(json.dumps(subscribe_payload))
        print("Connected! Streaming 5 fully-parsed AIS messages...\n")

        message_count = 0
        max_messages = 5

        try:
            async for raw_message in websocket:
                data = json.loads(raw_message)
                msg_type = data.get("MessageType")

                if msg_type == "SubscriptionConfirmation":
                    print("Subscription confirmed. Listening...\n")
                    continue

                message_count += 1

                # Parse and clean message contents
                extracted_data = extract_comprehensive_ais_data(data)

                print(
                    f"=================== AIS MESSAGE #{message_count} [{msg_type}] ==================="
                )
                print(json.dumps(extracted_data, indent=2, ensure_ascii=False))
                print(
                    "=================================================================================\n"
                )

                if message_count >= max_messages:
                    print("Successfully extracted 5 complete messages!")
                    break

        except websockets.exceptions.ConnectionClosed as e:
            print(f"Connection closed by server: {e}")


await run_full_extraction()

Connecting to wss://stream.aisstream.io/v0/stream...
Connected! Streaming 5 fully-parsed AIS messages...

Subscription confirmed. Listening...

=================== AIS MESSAGE #1 [PositionReport] ===================
{
  "message_type": "PositionReport",
  "mmsi": 373875000,
  "ship_name_meta": "OCEAN ULTRA",
  "latitude": 1.23096,
  "longitude": 103.96145,
  "timestamp_utc": "2026-10-10 20:54:40.225263625 +0000 UTC",
  "speed_over_ground_knots": 10.2,
  "course_over_ground_degrees": 83.2,
  "true_heading_degrees": 83,
  "navigational_status_code": 0,
  "rate_of_turn": 0,
  "position_accuracy": false,
  "raim_flag": false,
  "valid_position": true
}

=================== AIS MESSAGE #2 [PositionReport] ===================
{
  "message_type": "PositionReport",
  "mmsi": 574379000,
  "ship_name_meta": "GREAT OCEAN",
  "latitude": 1.24422,
  "longitude": 103.79593,
  "timestamp_utc": "2026-10-10 20:54:40.728579617 +0000 UTC",
  "speed_over_ground_knots": 0,
  "course_over_ground_degrees": 3